# MuleNet-Z — Enclave V2 Privacy Demo

**Purpose:** Demonstrate application-level privacy controls for cross-bank
transaction-risk analysis on IBM LinuxONE.

This prototype demonstrates encrypted bank submissions, bank-specific
encryption keys, a simulated attestation gate, encrypted bank-specific
alerts, and isolation checks.

**Important:** Attestation and enclave execution are software simulations.
This notebook does not establish hardware-backed Secure Execution or
production remote attestation.


## 1. Configure the demo

Use the sample dataset and one day of transactions to limit memory use.
The current fallback model is the combined XGBoost score file.


In [1]:
import sys
import os
from pathlib import Path
ROOT = Path.home() / 'shared'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pathlib import Path
import os
import time
import pandas as pd

from mulenet.enclave import (
    INSTS,
    make_keyring,
    bank_submit,
    measure_enclave,
    release_keys,
    enclave_run,
    verify_isolation,
    read_my_alerts,
)

ROOT = Path.home() / "shared"
SCORES = ROOT / "results/scores_sample/gfpxgb_combined.csv"
OUT = ROOT / "results/alerts_enc"

assert SCORES.exists(), f"Score file not found: {SCORES}"

print("Project:", ROOT)
print("Score source:", SCORES.name)
print("Banks:", list(INSTS))
print("Sample mode: enabled")
print("Transaction window: 1 day")


Project: /home/ibm-user/shared
Score source: gfpxgb_combined.csv
Banks: ['B1', 'B2', 'B3', 'B4', 'B5']
Sample mode: enabled
Transaction window: 1 day


## 2. Create bank-specific keys and encrypted submissions

The keyring is local and must never be committed to Git or shared publicly.
Only print its path and permissions—not the key contents.


In [2]:
ring = make_keyring()

keyring_path = ROOT / "secrets/keyring.json"
if keyring_path.exists():
    print("Keyring exists:", keyring_path)
    print("Permissions:", oct(keyring_path.stat().st_mode & 0o777))

start = time.perf_counter()
subs = {
    bank: bank_submit(bank, ring, sample=True, days=1)
    for bank in INSTS
}
submission_seconds = time.perf_counter() - start

print("Encrypted submissions created for:", list(subs.keys()))
print(f"Submission preparation time: {submission_seconds:.2f} seconds")


Keyring exists: /home/ibm-user/shared/secrets/keyring.json
Permissions: 0o600
Encrypted submissions created for: ['B1', 'B2', 'B3', 'B4', 'B5']
Submission preparation time: 12.32 seconds


## 3. Simulated attestation gate

The enclave measurement is a software-computed fingerprint. A deliberately
incorrect expected measurement should be rejected before keys are released.


In [3]:
measurement = measure_enclave()
print("Measured fingerprint:", measurement)

try:
    release_keys(ring, measurement, expected="0" * 64)
    raise AssertionError("Incorrect measurement was unexpectedly accepted")
except PermissionError as exc:
    print("Tampered measurement rejected:", exc)

keys = release_keys(ring, measurement, expected=measurement)
print("Matching measurement accepted; keys released for the demo.")


Measured fingerprint: 33fc583a510787d4d486318b9554d5b54d7a319b5de991c272ac2bae5b18477c
Tampered measurement rejected: Attestation failed: enclave image does not match the approved measurement
Matching measurement accepted; keys released for the demo.


## 4. Run Enclave V2

The pipeline decrypts bank submissions inside the software prototype,
joins the sample transaction data, applies combined model scores, and
writes separate encrypted alert files for each bank.


In [4]:
start = time.perf_counter()

result = enclave_run(
    subs,
    keys,
    scores_csv=str(SCORES),
    out_dir=OUT,
    hmac_key=ring["hmac"],
)

enclave_runtime = time.perf_counter() - start

print("Pipeline result:")
print(result)
print(f"Measured enclave_run wall time: {enclave_runtime:.2f} seconds")
print("Encrypted alert files:")
for bank in INSTS:
    alert_file = OUT / f"{bank}_alerts.enc"
    print(f"{bank}: exists={alert_file.exists()}")


Pipeline result:
{'files': {'B1': '/home/ibm-user/shared/results/alerts_enc/B1_alerts.enc', 'B2': '/home/ibm-user/shared/results/alerts_enc/B2_alerts.enc', 'B3': '/home/ibm-user/shared/results/alerts_enc/B3_alerts.enc', 'B4': '/home/ibm-user/shared/results/alerts_enc/B4_alerts.enc', 'B5': '/home/ibm-user/shared/results/alerts_enc/B5_alerts.enc'}, 'joined_transactions': 1114921, 'joined_accounts': 510772, 'decrypt_seconds': 0.8466738439965411, 'total_seconds': 6.70957827699749}
Measured enclave_run wall time: 7.03 seconds
Encrypted alert files:
B1: exists=True
B2: exists=True
B3: exists=True
B4: exists=True
B5: exists=True


## 5. Verify bank isolation

Each bank should be able to read its own alert file. The isolation check
should report that the alerts map only to that bank's accounts and that
other banks' keys cannot decrypt its file.


In [5]:
report = verify_isolation(OUT, ring, sample=True)
print("Isolation report:")
print(report)


Isolation report:
  bank  alerts  all_own_accounts  other_banks_could_open
0   B1     183              True                       0
1   B2     212              True                       0
2   B3     208              True                       0
3   B4     207              True                       0
4   B5     212              True                       0


## 6. Preview one bank's own alerts

This preview uses the B1 bank-specific read function. Raw account identifiers
are not included in the displayed preview; the bank-specific pseudonym and
risk evidence are shown instead.


In [8]:
preview = read_my_alerts("B1", ring, OUT, sample=True)
cols = ["account_pseudonym", "score", "rank",
        "evidence.senders_from_members", "evidence.receivers_at_members"]
show = preview[[c for c in cols if c in preview.columns]].head(10).rename(columns={
    "account_pseudonym": "pseudonymised account",
    "evidence.senders_from_members": "member banks sending to it",
    "evidence.receivers_at_members": "member banks it sends to"})
print("B1 alert count:", len(preview))
display(show)

B1 alert count: 183


,pseudonymised account,score,rank,member banks sending to it,member banks it sends to
0,13230309098c60531db1df76,0.9921,1,0,5
1,9a9718bdd3eeb4e76e812625,0.9861,2,2,5
2,984c9934617796debbb8c510,0.9796,3,2,5
3,a86ff04f429219d5c199f6e6,0.9741,4,1,1
4,3a1afb9919fda60d390b2dd5,0.9734,5,1,1
5,a7655a77f3e8009462a80c2b,0.9727,6,1,1
6,2a90e0558920f27bfaa4b32c,0.9718,7,1,1
7,de3c7554efb692c9649c4088,0.9707,8,1,2
8,6b71e965919dca543c6c3f53,0.9703,9,1,2
9,851994109a7fcfca7d979466,0.9700,10,2,1


## 7. Demo summary and limitations

### Demonstrated
- Bank-specific encrypted submissions and encrypted alert files.
- A separate bank key design and stable consortium pseudonyms.
- Rejection of an incorrect software-computed measurement.
- Bank-level alert isolation checks.
- Risk ranking from the combined XGBoost score file.

### Limitations
- The enclave and attestation mechanism are simulated in software.
- This is not proof of IBM Secure Execution hardware isolation or
  production remote attestation.
- This run uses the combined XGBoost scores, not GNN scores.
- Alerts are selected using the implementation's top-1% per-bank
  test-split budget; they are risk candidates, not confirmed fraud.
- Runtime figures describe the measured operations only, not the entire
  end-to-end project workflow.
